# 06 · Shallow ConvNCF on Yelp

Thesis §5.6.2 · Table 15 · Figures 24–25

MF-BPR pretraining (5 epochs), then ConvNCF (32, 64, 64) with BPR loss and 1 negative per positive,
30 epochs, validation every 3 epochs.

> **Note.** This local copy is a later edit of the Kaggle notebook. It uses `Adam(0.05)` and a fixed
> L2 = 1e-4. The run reported in the thesis used `Adagrad(0.05)` with the L2 grid search described in
> §5.6.2. A stray `eval_batch_size=` argument, which `eval_hr_ndcg` does not accept, was removed from
> the training loop so the notebook runs.

In [ ]:
# Folder containing ml-1m.ratings.dat and yelp.rating (see data/README.md).
# On Kaggle, point this at the attached dataset, e.g. "/kaggle/input/<dataset-name>".
DATA_DIR = "../data"

Plan:
1) Pretrain MF BPR for 5 epochs
2) Add embeddings to the ConvNCF model
3) Run for one epoch with all regularisations = 0
4) Run a hyper parameter search for l2_emb, l2_conv, l2_out, each for values: [10−3,10−1,10] (9 setups) for 30 epochs max
5) Pick best weights based on the hr and ndcg of the validation
6) But, if pairwise accuracy < 0.01  => stop for this setup
7) Calculate hr and ndcg after every 2 epochs, if both of them decreases stop for this setup, or if previous setups the results are better
8) Run on a test set HR@K and NDCG@K test, for k =5,10,20 on test set

In [ ]:
import pandas as pd
import os
import sys

def read_yelp_rating_file(file_path):
    # Check if the file exists
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' does not exist.")
        sys.exit(1)

    try:
        # Read the file into a DataFrame (auto-detects whitespace or tab)
        df = pd.read_csv(
            file_path,
            sep=r'\s+',            # handles both tabs and spaces
            header=None,
            names=['user_id', 'item_id', 'rating', 'timestamp'],
            engine='python'        # allows regex separator
        )

        # Display basic information about the DataFrame
        print("✅ Data successfully loaded.\n")
        print("DataFrame Info:")
        print(df.info())
        print("\nFirst 5 rows of the DataFrame:")
        print(df.head())

        return df

    except Exception as e:
        print(f" Error reading file '{file_path}': {e}")
        sys.exit(1)

yelp_file_path = f"{DATA_DIR}/yelp.rating"
yelp_df = read_yelp_rating_file(yelp_file_path)
df = yelp_df

✅ Data successfully loaded.

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731671 entries, 0 to 731670
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    731671 non-null  int64  
 1   item_id    731671 non-null  int64  
 2   rating     731671 non-null  float64
 3   timestamp  731671 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 22.3 MB
None

First 5 rows of the DataFrame:
   user_id  item_id  rating   timestamp
0        0        0     4.0  1329148800
1        0        0     4.0  1337011200
2        0        1     4.0  1335888000
3        0        2     4.0  1379260800
4        0        3     2.0  1367856000


In [ ]:
import random, math
from functools import lru_cache
from collections import defaultdict
#HR (hit ratio)
# ---- Metrics ----
def HR_from_ranked_items(ranked_item_ids, true_item, K):
    # ranked_item_ids: 1D array/list of item ids sorted by predicted score desc
    return 1.0 if true_item in ranked_item_ids[:K] else 0.0

def NDCG_from_ranked_items(ranked_item_ids, true_item, K):
    topk = ranked_item_ids[:K]
    for rank, iid in enumerate(topk):
        if iid == true_item:
            return 1.0 / math.log2((rank + 1) + 1)
    return 0.0


# ---- Evaluation (fixed) ----
def eval_hr_ndcg(
    interaction_array,   # shape (num_users, num_items), aligned to label-encoded ids
    test_df,             # label-encoded columns: user_id, item_id
    model,               # scorer model: inputs [user_id, item_id] -> score
    number_of_rec_items=10,
    max_test_size=10_000,
    neg_sample_size=100,
    seed=42,
):
    rng = np.random.default_rng(seed)
    n_users, n_items = interaction_array.shape

    # Precompute user positive sets once (fast membership checks)
    user_pos = [None] * n_users
    for u in range(n_users):
        user_pos[u] = set(np.flatnonzero(interaction_array[u]))

    users = test_df['user_id'].to_numpy(dtype=np.int32)
    items = test_df['item_id'].to_numpy(dtype=np.int32)

    num_hits = 0.0
    sum_ndcg = 0.0
    num_test = 0

    for u, pos_item in zip(users, items):
        if num_test >= max_test_size:
            break
        if u < 0 or u >= n_users or pos_item < 0 or pos_item >= n_items:
            continue

        pos_set = user_pos[u]

        # If user interacted with almost everything, skip
        if len(pos_set) >= n_items - 1:
            continue

        # Sample negatives with rejection sampling (avoids set-difference materialization)
        k = min(neg_sample_size, n_items - len(pos_set) - 1)
        if k <= 0:
            continue

        negs = np.empty(k, dtype=np.int32)
        filled = 0
        while filled < k:
            draw = rng.integers(0, n_items, size=(k - filled) * 2, dtype=np.int32)
            for j in draw:
                if (j != pos_item) and (j not in pos_set):
                    negs[filled] = j
                    filled += 1
                    if filled == k:
                        break

        cand_items = np.concatenate([negs, np.array([pos_item], dtype=np.int32)])
        cand_users = np.full(cand_items.shape[0], u, dtype=np.int32)

        preds = model.predict([cand_users, cand_items], verbose=0).reshape(-1)

        # Rank candidates by score desc
        ranked_idx = np.argsort(-preds)
        ranked_items = cand_items[ranked_idx]

        num_hits += HR_from_ranked_items(ranked_items, pos_item, number_of_rec_items)
        sum_ndcg += NDCG_from_ranked_items(ranked_items, pos_item, number_of_rec_items)
        num_test += 1

    if num_test == 0:
        return 0.0, 0.0
    return num_hits / num_test, sum_ndcg / num_test

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
#Check that every user has >=20 rankings

user_counts = df.groupby("user_id").size()
print("Minimum number of ratings per user:", user_counts.min())

# Sort the data by user, then by timestamp (ascending)
df = df.sort_values(['user_id', 'timestamp'])

# We will collect train rows and test rows in separate lists
train_list = []
test_list  = []

# Group by each user
for user_id, group in df.groupby('user_id'):
    # Convert group to list (or keep as DF if you like)
    user_history = group.values  # or group.to_numpy()

    # The last row of this user’s history is the "most recent" (test instance)
    test_interaction = user_history[-1]
    
    # All preceding rows are training data
    train_interactions = user_history[:-1]

    # Add them to our overall lists
    for row in train_interactions:
        train_list.append(row)
    test_list.append(test_interaction)

# Convert lists back to DataFrame
train_df = pd.DataFrame(train_list, columns=df.columns)
test_df  = pd.DataFrame(test_list,  columns=df.columns)
train_df = train_df.sort_values(['user_id', 'timestamp'])

val_df = train_df.groupby('user_id').tail(1).copy()   # 1 row per user
train_df_cv = train_df.drop(val_df.index).copy()

train_df = train_df_cv
print("Training size:", len(train_df))
print("Testing size: ", len(test_df))
# --- label-encode user/item to 0..n-1 (fit on TRAIN_CV + VAL + TEST) ---
ue, ie = LabelEncoder(), LabelEncoder()

all_ids = pd.concat([
    train_df[['user_id','item_id']],   # TRAIN_CV
    val_df[['user_id','item_id']],     # VAL  <-- include this
    test_df[['user_id','item_id']]     # TEST
], axis=0)

ue.fit(all_ids['user_id'])
ie.fit(all_ids['item_id'])

train_df['user_id'] = ue.transform(train_df['user_id'])
train_df['item_id'] = ie.transform(train_df['item_id'])
val_df['user_id']   = ue.transform(val_df['user_id'])    # <-- encode VAL
val_df['item_id']   = ie.transform(val_df['item_id'])    # <-- encode VAL
test_df['user_id']  = ue.transform(test_df['user_id'])
test_df['item_id']  = ie.transform(test_df['item_id'])

num_users = len(ue.classes_)
num_items = len(ie.classes_)

# --- per-user positive sets (from TRAIN_CV only, to avoid leakage) ---
user_pos = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
all_items = np.arange(num_items, dtype=np.int32)

def make_bpr_triplets(user_pos, num_items, num_neg, seed=42):
    rng = np.random.default_rng(seed)
    users, pos_items, neg_items = [], [], []
    for u, pos_set in user_pos.items():
        pos_list = list(pos_set)
        for i in pos_list:
            for _ in range(num_neg):
                j = rng.integers(0, num_items)
                # resample until we hit a true negative
                while j in pos_set:
                    j = rng.integers(0, num_items)
                users.append(u); pos_items.append(i); neg_items.append(j)
    users = np.array(users, dtype=np.int32)
    pos_items = np.array(pos_items, dtype=np.int32)
    neg_items = np.array(neg_items, dtype=np.int32)
    return users, pos_items, neg_items


import tensorflow as tf
class BPRSequence(tf.keras.utils.Sequence):
    def __init__(self, user_pos, num_items, batch_size=1024, num_neg=1, seed=42):
        self.user_pos = user_pos
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_neg = num_neg
        self.rng = np.random.default_rng(seed)
        self._rebuild()

    def _rebuild(self):
        self.u, self.i, self.j = make_bpr_triplets(
            self.user_pos,
            self.num_items,
            self.num_neg,
            seed=self.rng.integers(1_000_000)
            )
        self.n = self.u.shape[0]
        self.order = np.arange(self.n)
        self.rng.shuffle(self.order)

    def __len__(self):
        return int(np.ceil(self.n / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end   = min(start + self.batch_size, self.n)
        bi    = self.order[start:end]

        X = {
            'user_id':     self.u[bi].astype(np.int32),
            'pos_item_id': self.i[bi].astype(np.int32),
            'neg_item_id': self.j[bi].astype(np.int32),
        }
        y = np.ones((len(bi), 1), dtype=np.float32)
        return X, y

    def on_epoch_end(self):
        self._rebuild()

# Build BPR sequence once and reuse everywhere (auto-resamples negatives each epoch)
train_seq = BPRSequence(user_pos, num_items, batch_size=256, num_neg=1)


Minimum number of ratings per user: 10
Training size: 680317
Testing size:  25677


2025-11-10 18:06:35.361427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762797995.563264      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762797995.617968      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers


def build_mf_scorer(num_users, num_items, emb_size):
    u_in = layers.Input(shape=(), dtype='int32', name='mf_user_id')
    i_in = layers.Input(shape=(), dtype='int32', name='mf_item_id')

    uE = layers.Embedding(num_users, emb_size, name='mf_user_emb')(u_in)
    iE = layers.Embedding(num_items, emb_size, name='mf_item_emb')(i_in)

    score = layers.Dot(axes=-1, normalize=False, name='mf_dot')([uE, iE])
    return Model([u_in, i_in], score, name='MFScorer')



def build_bpr_model(scorer: tf.keras.Model):
    u_in  = layers.Input(shape=(), dtype='int32', name='user_id')
    i_pos = layers.Input(shape=(), dtype='int32', name='pos_item_id')
    i_neg = layers.Input(shape=(), dtype='int32', name='neg_item_id')

    s_pos = scorer([u_in, i_pos])
    s_neg = scorer([u_in, i_neg])
    diff  = layers.Subtract(name='logit_diff')([s_pos, s_neg])  # (B, 1) logits

    return Model([u_in, i_pos, i_neg], diff, name='ConvNCF_BPR')




def build_convncf_scorer(
        num_users,
        num_items,
        emb_size,
        conv_filters,
        kernel_size,
        dense_units,
        l2_emb=0.0,
        l2_conv=0.0,
        l2_out=0.0
    ):
    L2e = (lambda v: regularizers.l2(v) if v and v > 0 else None)

    user_input = layers.Input(shape=(), dtype='int32', name='sc_user_id')
    item_input = layers.Input(shape=(), dtype='int32', name='sc_item_id')

    user_emb = layers.Embedding(
        input_dim=num_users, output_dim=emb_size,
        embeddings_regularizer=L2e(l2_emb),
        name='user_embedding')(user_input)

    item_emb = layers.Embedding(
        input_dim=num_items, output_dim=emb_size,
        embeddings_regularizer=L2e(l2_emb),
        name='item_embedding')(item_input)

    u_exp = layers.Lambda(lambda x: tf.expand_dims(x, -1), name='u_expand')(user_emb)  # (B, emb, 1)
    v_exp = layers.Lambda(lambda x: tf.expand_dims(x,  1), name='v_expand')(item_emb)  # (B, 1, emb)
    E = layers.Multiply(name='outer_prod')([u_exp, v_exp])                              # (B, emb, emb)
    E = layers.Lambda(lambda x: tf.expand_dims(x, -1), name='add_channel')(E)          # (B, emb, emb, 1)

    x = E
    for idx, f in enumerate(conv_filters, 1):
        x = layers.Conv2D(
            filters=f, kernel_size=(kernel_size, kernel_size),
            activation='relu', padding='same',
            kernel_regularizer=L2e(l2_conv),
            name=f'conv{idx}')(x)
        x = layers.MaxPooling2D(pool_size=(2, 2), name=f'pool{idx}')(x)

    x = layers.Flatten(name='flatten')(x)
    x = layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=L2e(l2_conv), name='dense')(x)

    score = layers.Dense(
        1, activation=None, name='score',
        kernel_regularizer=L2e(l2_out)
    )(x)

    return Model(inputs=[user_input, item_input], outputs=score, name='ConvNCFScorer')




def pairwise_accuracy(y_true, logit_diff):
    return tf.reduce_mean(tf.cast(logit_diff > 0.0, tf.float32))


In [ ]:
def pretrain_MF(BATCH_SIZE, NEG_PER_POS, MF_EPOCHS, emb_size=64, seed=42):
    # Build MF scorer
    mf_scorer = build_mf_scorer(num_users, num_items, emb_size=emb_size)
    mf_bpr    = build_bpr_model(mf_scorer)

    mf_bpr.compile(
        optimizer=tf.keras.optimizers.Adagrad(0.05),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=[pairwise_accuracy]
    )



    mf_bpr.fit(train_seq, epochs=MF_EPOCHS, verbose=1)

    P = mf_scorer.get_layer('mf_user_emb').get_weights()[0]
    Q = mf_scorer.get_layer('mf_item_emb').get_weights()[0]
    return P, Q


I0000 00:00:1762798036.956765      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1762798045.430784     115 service.cc:148] XLA service 0x7960f4008ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762798045.431660     115 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1762798046.156050     115 cuda_dnn.cc:529] Loaded cuDNN version 90300


   63/10135 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - loss: 0.6932 - pairwise_accuracy: 0.4931

I0000 00:00:1762798051.896139     115 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10135/10135 ━━━━━━━━━━━━━━━━━━━━ 45s 4ms/step - loss: 0.6932 - pairwise_accuracy: 0.5006
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 0.6931 - pairwise_accuracy: 0.5065
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 0.6930 - pairwise_accuracy: 0.5121
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 30s 3ms/step - loss: 0.6930 - pairwise_accuracy: 0.5168
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - loss: 0.6929 - pairwise_accuracy: 0.5225


In [ ]:
import numpy as np
import tensorflow as tf
import math

# ============================================================
# 0) Build VAL interaction matrix (from TRAIN only), aligned
# ============================================================
val_inter_mat = (
    train_df.assign(interaction=1)
            .pivot_table(index='user_id', columns='item_id',
                         values='interaction', fill_value=0)
).reindex(index=np.arange(num_users),
          columns=np.arange(num_items),
          fill_value=0)

val_inter_arr = val_inter_mat.values

# For TEST, negatives should be sampled excluding TRAIN interactions
train_inter_mat = (
    train_df.assign(interaction=1)
            .pivot_table(index='user_id', columns='item_id',
                         values='interaction', fill_value=0)
).reindex(index=np.arange(num_users),
          columns=np.arange(num_items),
          fill_value=0)
train_inter_arr = train_inter_mat.values

# ============================================================
# Config (single run)
# ============================================================
BATCH_SIZE     = 256
NEG_PER_POS    = 4
K_VAL          = 10
MAX_EPOCHS     = 30
MF_EPOCHS      = 5
VAL_NEG_SIZE   = 100
TEST_NEG_SIZE  = 99
EVAL_EVERY     = 3

# Architecture
emb_size     = 64
conv_filters = (32, 64, 64)   # strongly recommend starting here
kernel_size  = 2
dense_units  = 128

# Regularization (single choice)
L2_EMB  = 1e-4
L2_CONV = 1e-4
L2_OUT  = 1e-4

# Optimizer (Adam is the easiest way to avoid “flat loss” issues)
OPT = tf.keras.optimizers.Adam(0.05)

# ===========================================================
# ============================================================
# 2) MF pretrain (expects your pretrain_MF returns P, Q)
# ============================================================
print("\n=== Step 1: Pretrain MF-BPR ===")
P, Q = pretrain_MF(
    BATCH_SIZE=BATCH_SIZE,
    NEG_PER_POS=NEG_PER_POS,
    MF_EPOCHS=MF_EPOCHS,
    emb_size=emb_size
)
print("MF pretraining done. P:", P.shape, "Q:", Q.shape)

# ============================================================
# 3) ConvNCF warmup: no-reg scorer, 1 epoch
# ============================================================
print("\n=== Step 2: ConvNCF warmup (no-reg, 1 epoch) ===")
scorer_no_reg = build_convncf_scorer(
    num_users=num_users,
    num_items=num_items,
    emb_size=emb_size,
    conv_filters=conv_filters,
    kernel_size=kernel_size,
    dense_units=dense_units,
    l2_emb=0.0,
    l2_conv=0.0,
    l2_out=0.0
)
scorer_no_reg.get_layer('user_embedding').set_weights([P])
scorer_no_reg.get_layer('item_embedding').set_weights([Q])

bpr_no_reg = build_bpr_model(scorer_no_reg)
bpr_no_reg.compile(
    optimizer=OPT,
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[pairwise_accuracy],
)
bpr_no_reg.fit(train_seq, epochs=1, verbose=1)

# ============================================================
# 4) Final scorer with chosen L2, init from warmup
# ============================================================
print("\n=== Step 3: Train final ConvNCF with fixed regularization ===")
scorer = build_convncf_scorer(
    num_users=num_users,
    num_items=num_items,
    emb_size=emb_size,
    conv_filters=conv_filters,
    kernel_size=kernel_size,
    dense_units=dense_units,
    l2_emb=L2_EMB,
    l2_conv=L2_CONV,
    l2_out=L2_OUT
)
scorer.set_weights(scorer_no_reg.get_weights())

bpr_model = build_bpr_model(scorer)
bpr_model.compile(
    optimizer=OPT,
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[pairwise_accuracy],
)

best = {'ndcg': -1.0, 'hr': 0.0, 'epoch': -1, 'weights': None}
train_loss_hist, train_pair_hist = [], []
val_epochs, val_hr_hist, val_ndcg_hist = [], [], []

for ep in range(1, MAX_EPOCHS + 1):
    hist = bpr_model.fit(train_seq, epochs=1, verbose=1)
    loss = float(hist.history['loss'][0])
    pair_acc = float(hist.history['pairwise_accuracy'][0])

    train_loss_hist.append(loss)
    train_pair_hist.append(pair_acc)
    print(f"Epoch {ep:02d} | loss={loss:.4f} | pairwise_acc={pair_acc:.4f}")

    if ep % EVAL_EVERY == 0:
        hr, ndcg = eval_hr_ndcg(   # use your faster evaluator if you added it
            interaction_array=val_inter_arr,
            test_df=val_df,
            model=scorer,
            number_of_rec_items=K_VAL,
            max_test_size=10_000,
            neg_sample_size=VAL_NEG_SIZE,
            seed=42
        )
        print(f"[VAL] epoch {ep:02d} — HR@{K_VAL}={hr:.4f}  NDCG@{K_VAL}={ndcg:.4f}")

        val_epochs.append(ep)
        val_hr_hist.append(hr)
        val_ndcg_hist.append(ndcg)

        if (ndcg > best['ndcg']) or (math.isclose(ndcg, best['ndcg']) and hr > best['hr']):
            best.update({'ndcg': ndcg, 'hr': hr, 'epoch': ep, 'weights': scorer.get_weights()})
            print("Saved new best weights.")

print(f"\nBest VAL: epoch {best['epoch']:02d} — HR@{K_VAL}={best['hr']:.4f}  NDCG@{K_VAL}={best['ndcg']:.4f}")

# ============================================================
# 5) Final TEST evaluation using best weights
# ============================================================
if best['weights'] is None:
    print("No best weights found; skipping TEST.")
else:
    scorer.set_weights(best['weights'])
    for K_test in [5, 10, 20]:
        hr_t, ndcg_t = eval_hr_ndcg(
            interaction_array=train_inter_arr,   # exclude TRAIN positives when sampling negatives
            test_df=test_df,
            model=scorer,
            number_of_rec_items=K_test,
            max_test_size=10_000,
            neg_sample_size=TEST_NEG_SIZE,
            seed=42
        )
        print(f"[TEST] HR@{K_test}={hr_t:.4f}  NDCG@{K_test}={ndcg_t:.4f}")


10135/10135 ━━━━━━━━━━━━━━━━━━━━ 265s 25ms/step - loss: 0.6932 - pairwise_accuracy: 0.5126
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 250s 24ms/step - loss: 1.8735 - pairwise_accuracy: 0.5135
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 244s 24ms/step - loss: 0.6951 - pairwise_accuracy: 0.0690
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 242s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 242s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 242s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 242s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 240s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 241s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 241s 24ms/step - loss: 0.6932 - pairwise_accuracy: 0.0000e+00
10135/10135 ━━━━━━━━━━━━━━━━━━━━ 241s 24ms/step - loss: 0.6932

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----- Training loss and pairwise accuracy vs epoch -----
epochs = np.arange(1, len(train_loss_hist) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss_hist, label='train_loss')
plt.plot(epochs, train_pair_hist, label='pairwise_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('ConvNCF: Training Loss & Pairwise Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# ----- (optional) Loss vs Pairwise Accuracy scatter -----
plt.figure(figsize=(6, 5))
plt.scatter(train_pair_hist, train_loss_hist)
plt.xlabel('Pairwise Accuracy')
plt.ylabel('Training Loss')
plt.title('ConvNCF: Loss vs Pairwise Accuracy')
plt.tight_layout()
plt.show()

# ----- Validation HR@K and NDCG@K vs epoch -----
if len(val_epochs) > 0:
    plt.figure(figsize=(8, 5))
    plt.plot(val_epochs, val_hr_hist, marker='o', label=f'val_HR@{K_VAL}')
    plt.plot(val_epochs, val_ndcg_hist, marker='s', label=f'val_NDCG@{K_VAL}')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.title(f'ConvNCF: Validation HR@{K_VAL} and NDCG@{K_VAL}')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No validation points recorded (check EVAL_EVERY / MAX_EPOCHS).")
